# Tyson Cell-Cycle – NNSE Neutral-Set Sampling + Jackknife

Self-contained notebook for the 6-state Tyson ODE model.  
Runs the NNSE algorithm to map the neutral set, then optionally runs a  
jackknife stability diagnostic over independent full runs.

**Varied parameters** (6): `k1_aa_over_CT`, `k3_CT`, `k4`, `k4prime`, `k6`, `k7`  
**Fixed**: `k2=0`, `k5_minusP=0`, `k8_minusP=100`, `k9=50`, `CT=1`  
**Observables**: `YT/CT` and `M/CT`

In [ ]:
%matplotlib inline

from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# NNSE
SIGMA            = 0.01    # Gaussian mutation SD in normalised space
MAX_STEPS        = 10000   # safety cap
MIN_FILLED       = 29      # run until this many bins occupied (out of N_VEC)
STABILIZATION    = 2500    # extra steps after MIN_FILLED reached
BURN_IN_FALLBACK = 5000    # start tracking swaps at this step if MIN_FILLED not yet reached
SEED             = 42

# Bin thresholds: 30-bin logspace(0.1, 15) + 4 extra lower bins prepended
MAX_VALUE   = 15.0
N_BASE      = 30
N_EXTRA     = 4
base        = np.logspace(np.log10(0.1), np.log10(MAX_VALUE), N_BASE + 1)
ratio       = base[1] / base[0]
extra_low   = base[0] / (ratio ** np.arange(N_EXTRA, 0, -1))
bin_thresholds = np.concatenate([extra_low, base])
bin_thresholds[-1] = 1000.0
N_VEC = len(bin_thresholds)

# Simulation
T_END    = 500.0
N_POINTS = 501
t_eval   = np.linspace(0.0, T_END, N_POINTS)

# Jackknife (set True to run reproducibility analysis)
JACKKNIFE = False
N_RUNS    = 20
SEED0     = 0

# Output directories
PLOT_DIR   = Path("..") / "plots" / "TysonNNSE"
RESULT_DIR = Path("..") / "results" / "nnse"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Bins: {N_VEC}  (thresholds {bin_thresholds[0]:.3e} … {bin_thresholds[-1]:.3e})")
print(f"NNSE: sigma={SIGMA}, max_steps={MAX_STEPS}, min_filled={MIN_FILLED}, stab={STABILIZATION}")
print(f"Jackknife: {'ENABLED' if JACKKNIFE else 'DISABLED'}")

In [ ]:
# ── Tyson ODE model ───────────────────────────────────────────────────────────

# Wildtype parameters
P0 = {
    "k1_aa_over_CT": 0.015,
    "k2":            0.0,     # fixed (zero)
    "k3_CT":         200.0,
    "k4":            180.0,
    "k4prime":       0.018,
    "k5_minusP":     0.0,     # fixed (zero)
    "k6":            1.0,
    "k7":            0.6,
    "k8_minusP":     100.0,   # fixed
    "k9":            50.0,    # fixed
    "CT":            1.0,     # fixed
}

PARAM_NAMES = ["k1_aa_over_CT", "k3_CT", "k4", "k4prime", "k6", "k7"]
p0_vec = np.array([P0[name] for name in PARAM_NAMES])
CT = P0["CT"]

# Initial state: [C2, CP, pM, M, Y, YP]
Y0_STATE = np.array([0.9, 0.05, 0.0, 0.005, 0.3, 0.0])


def F_M(M: float, p: dict) -> float:
    return p["k4prime"] + p["k4"] * (M / p["CT"]) ** 2


def f_rhs(t: float, x: np.ndarray, p: dict) -> np.ndarray:
    C2, CP, pM, M, Y, YP = x
    k3 = p["k3_CT"] / p["CT"]
    k1 = p["k1_aa_over_CT"] * p["CT"]
    dC2 = p["k6"] * M - p["k8_minusP"] * C2 + p["k9"] * CP
    dCP = -k3 * CP * Y + p["k8_minusP"] * C2 - p["k9"] * CP
    dpM = k3 * CP * Y - pM * F_M(M, p) + p["k5_minusP"] * M
    dM  = pM * F_M(M, p) - p["k5_minusP"] * M - p["k6"] * M
    dY  = k1 - p["k2"] * Y - k3 * CP * Y
    dYP = p["k6"] * M - p["k7"] * YP
    return np.array([dC2, dCP, dpM, dM, dY, dYP])


def simulate(p_dict: dict, t_ev: np.ndarray):
    sol = solve_ivp(
        lambda t, x: f_rhs(t, x, p_dict),
        (t_ev[0], t_ev[-1]),
        Y0_STATE,
        method="BDF", t_eval=t_ev, rtol=1e-6, atol=1e-8,
    )
    return (sol.t, sol.y) if sol.success else None


def compute_obs(y: np.ndarray):
    """Return (YT/CT, M/CT) from state matrix (6, n_t)."""
    C2, CP, pM, M, Y, YP = y
    YT = Y + YP + pM + M
    return YT / CT, M / CT


def vec_to_dict(vec: np.ndarray) -> dict:
    p = dict(P0)
    for name, val in zip(PARAM_NAMES, vec):
        p[name] = float(val)
    return p

In [ ]:
# ── Reference simulation ───────────────────────────────────────────────────────
ref = simulate(P0, t_eval)
if ref is None:
    raise RuntimeError("Wildtype reference simulation failed")
t_ref, y_ref = ref
YT_ref, M_ref = compute_obs(y_ref)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(t_ref, YT_ref, lw=1.5, color="steelblue")
axes[0].set_title("Reference: YT/CT"); axes[0].set_xlabel("Time"); axes[0].grid(alpha=0.3)
axes[1].plot(t_ref, M_ref, lw=1.5, color="tab:orange")
axes[1].set_title("Reference: M/CT"); axes[1].set_xlabel("Time"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_reference.png", dpi=150)
plt.show()

In [ ]:
# ── Objective function ────────────────────────────────────────────────────────

def objective(vec: np.ndarray) -> float:
    """Integrated squared difference in (YT/CT, M/CT) vs wildtype."""
    result = simulate(vec_to_dict(vec), t_eval)
    if result is None:
        return math.inf
    t, y = result
    if not np.all(np.isfinite(y)):
        return math.inf
    YT, M = compute_obs(y)
    YT0 = np.interp(t, t_ref, YT_ref)
    M0  = np.interp(t, t_ref, M_ref)
    return float(np.trapz((YT - YT0) ** 2 + (M - M0) ** 2, t))


# Quick sanity check: wildtype should give ~0
wt_score = objective(p0_vec)
print(f"Wildtype objective: {wt_score:.2e}  (expected ≈ 0)")

In [ ]:
# ── NNSE core functions ───────────────────────────────────────────────────────

def place_in_bin(value: float, thresholds: np.ndarray) -> int | None:
    if not math.isfinite(value):
        return None
    for idx, thr in enumerate(thresholds):
        if value <= thr:
            return idx
    return None


def nnse_step(
    xs: list,
    fs: list,
    p0: np.ndarray,
    thresholds: np.ndarray,
    rng: np.random.Generator,
    sigma: float,
) -> tuple[list, list, list]:
    n = len(thresholds)
    new_xs = [None] * n
    new_fs = [None] * n

    # Mutate each occupied position
    for i, (x, fx) in enumerate(zip(xs, fs)):
        if x is None:
            continue
        u     = x / (2.0 * p0)
        u_mut = (u + rng.normal(0.0, sigma, size=len(p0))) % 1.0
        x_mut = 2.0 * p0 * u_mut
        fx_mut = objective(x_mut)
        if fx_mut <= thresholds[i]:
            new_xs[i], new_fs[i] = x_mut, fx_mut
        else:
            new_xs[i], new_fs[i] = x.copy(), fx

    # Permutation: bubble lower-objective samples toward bin 0
    swaps: list[int] = []
    for i in range(n - 1, 0, -1):
        if new_fs[i] is not None and new_fs[i] <= thresholds[i - 1]:
            new_xs[i], new_xs[i - 1] = new_xs[i - 1], new_xs[i]
            new_fs[i], new_fs[i - 1] = new_fs[i - 1], new_fs[i]
            swaps.append(i)

    # Fill newly vacated positions
    for i in range(n):
        if new_xs[i] is not None:
            continue
        for _ in range(1000):
            cand  = 2.0 * p0 * rng.uniform(0.0, 1.0, size=len(p0))
            fcand = objective(cand)
            bidx  = place_in_bin(fcand, thresholds)
            if bidx is None:
                continue
            for pos in range(bidx, n):
                if new_xs[pos] is None:
                    new_xs[pos], new_fs[pos] = cand, fcand
                    break
            break

    return new_xs, new_fs, swaps

In [ ]:
# ── Run NNSE ──────────────────────────────────────────────────────────────────

def run_nnse(seed: int, verbose: bool = True) -> dict:
    rng = np.random.default_rng(seed)
    n   = N_VEC
    xs  = [None] * n
    fs  = [None] * n

    # Seed initial population
    attempts = 0
    while sum(x is not None for x in xs) == 0 and attempts < 10000:
        attempts += 1
        cand  = 2.0 * p0_vec * rng.uniform(0.0, 1.0, size=len(p0_vec))
        fcand = objective(cand)
        bidx  = place_in_bin(fcand, bin_thresholds)
        if bidx is not None:
            for pos in range(bidx, n):
                if xs[pos] is None:
                    xs[pos], fs[pos] = cand, fcand
                    break

    swap_count        = np.zeros(n)
    opportunity_count = np.zeros(n)
    best_history: list[float] = []
    neutral_points: list[np.ndarray] = []
    neutral_values: list[float]      = []
    convergence_step  = None
    remaining_stab    = None
    in_burnin         = False
    start = time.time()

    for step in range(MAX_STEPS):
        xs, fs, swaps = nnse_step(xs, fs, p0_vec, bin_thresholds, rng, SIGMA)
        filled  = sum(x is not None for x in xs)
        finite  = [fx for fx in fs if fx is not None and math.isfinite(fx)]
        best    = min(finite) if finite else math.inf
        best_history.append(best)

        # Activate burn-in when enough bins are filled, or at fallback step
        if not in_burnin and filled >= MIN_FILLED:
            convergence_step = step
            remaining_stab   = STABILIZATION
            in_burnin        = True
            if verbose:
                print(f"  Convergence at step {step}: {filled}/{n} filled. "
                      f"Running {STABILIZATION} stabilisation steps.")
        if not in_burnin and step >= BURN_IN_FALLBACK:
            in_burnin = True
            if verbose:
                print(f"  Fallback burn-in triggered at step {step}.")

        if in_burnin:
            swap_set = set(swaps)
            for i in range(1, n):
                if fs[i] is not None and fs[i - 1] is not None:
                    opportunity_count[i] += 1
                    if i in swap_set:
                        swap_count[i] += 1
            for x, fx in zip(xs, fs):
                if x is not None and fx is not None and fx <= bin_thresholds[0]:
                    neutral_points.append(x.copy())
                    neutral_values.append(float(fx))

        if remaining_stab is not None:
            remaining_stab -= 1
            if remaining_stab <= 0:
                break

        if verbose and (step == 0 or (step + 1) % max(1, MAX_STEPS // 20) == 0):
            elapsed = time.time() - start
            print(f"  step {step + 1:5d}: filled={filled}/{n}, best={best:.3e}, "
                  f"neutral={len(neutral_points)}, {elapsed:.0f}s")

    elapsed = time.time() - start
    volume_ratios = np.divide(
        swap_count, opportunity_count,
        out=np.full(n, np.nan),
        where=opportunity_count > 0,
    )
    neutral = (np.unique(np.array(neutral_points), axis=0)
               if neutral_points else np.empty((0, len(p0_vec))))

    if verbose:
        print(f"Done in {elapsed:.0f}s | neutral={len(neutral)} | "
              f"filled={sum(x is not None for x in xs)}/{n}")

    return {
        "xs":               xs,
        "fs":               fs,
        "volume_ratios":    volume_ratios,
        "swap_count":       swap_count,
        "opportunity_count": opportunity_count,
        "best_history":     np.array(best_history),
        "neutral_points":   neutral,
        "neutral_values":   np.array(neutral_values, dtype=float),
        "elapsed":          elapsed,
        "convergence_step": convergence_step,
    }


result = run_nnse(SEED)

In [ ]:
# ── NNSE result plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Volume ratios
valid = ~np.isnan(result["volume_ratios"][1:])
bidx  = np.where(valid)[0] + 1
axes[0].bar(bidx, result["volume_ratios"][1:][valid],
            color="steelblue", alpha=0.8, width=0.7)
axes[0].set_title("Volume ratios V[i−1]/V[i]")
axes[0].set_xlabel("Bin boundary i")
axes[0].set_ylabel("Swap rate")
axes[0].grid(alpha=0.3)

# 2. Best objective over steps
axes[1].semilogy(result["best_history"], lw=1, color="steelblue")
if result["convergence_step"] is not None:
    axes[1].axvline(result["convergence_step"], color="red", ls="--", lw=0.8, label="convergence")
    axes[1].legend()
axes[1].set_title("Best objective over steps")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Objective (log scale)")
axes[1].grid(alpha=0.3)

# 3. Neutral set parameter fold-changes
neutral = result["neutral_points"]
if len(neutral) > 0:
    ratios = neutral / p0_vec
    axes[2].boxplot([ratios[:, i] for i in range(len(PARAM_NAMES))],
                    labels=PARAM_NAMES, patch_artist=True,
                    boxprops=dict(facecolor="lightsteelblue", alpha=0.7))
    axes[2].axhline(1.0, color="red", lw=1.2, ls="--")
    axes[2].set_title(f"Neutral set ratios  (n={len(neutral)})")
    axes[2].set_ylabel("p / p₀")
    axes[2].tick_params(axis="x", rotation=30)
else:
    axes[2].text(0.5, 0.5, "No neutral points collected",
                 ha="center", va="center", transform=axes[2].transAxes)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "02_nnse_results.png", dpi=150)
plt.show()

In [ ]:
# ── Save NNSE result ──────────────────────────────────────────────────────────
import time as _time
stamp   = _time.strftime("%Y%m%d_%H%M%S")
out_npz = RESULT_DIR / f"tyson_nnse_{stamp}.npz"

final_x = np.array([x if x is not None else np.full(len(p0_vec), np.nan)
                    for x in result["xs"]])
final_f = np.array([fx if fx is not None else np.nan for fx in result["fs"]], dtype=float)

np.savez_compressed(
    out_npz,
    neutral_points           = result["neutral_points"],
    neutral_objective_values = result["neutral_values"],
    final_population         = final_x,
    final_objective_values   = final_f,
    p0                       = p0_vec,
    parameter_names          = np.array(PARAM_NAMES, dtype=object),
    bin_thresholds           = bin_thresholds,
    volume_ratios            = result["volume_ratios"],
    swap_count               = result["swap_count"],
    opportunity_count        = result["opportunity_count"],
    best_history             = result["best_history"],
    reference_time           = t_ref,
    reference_YT             = YT_ref,
    reference_M              = M_ref,
)
print(f"Saved: {out_npz}")

In [ ]:
# ── Jackknife stability diagnostic (runs only if JACKKNIFE = True) ────────────

if not JACKKNIFE:
    print("Jackknife disabled. Set JACKKNIFE = True in the config cell to run.")
else:
    print(f"Running {N_RUNS} independent NNSE simulations for jackknife …")
    all_volume_ratios: list[np.ndarray] = []

    for i in range(N_RUNS):
        seed_i = SEED0 + i
        t0 = time.time()
        r  = run_nnse(seed_i, verbose=False)
        all_volume_ratios.append(r["volume_ratios"])
        print(f"  Run {i + 1:2d}/{N_RUNS}  seed={seed_i}  "
              f"elapsed={time.time() - t0:.0f}s  "
              f"neutral={len(r['neutral_points'])}")

    # ── Jackknife statistics ─────────────────────────────────────────────────
    mat = np.array(all_volume_ratios)   # (N_RUNS, N_VEC)
    n_r = N_RUNS

    theta_hat = np.nanmean(mat, axis=0)
    theta_j   = np.array([np.nanmean(np.delete(mat, i, axis=0), axis=0)
                          for i in range(n_r)])
    theta_bar = np.nanmean(theta_j, axis=0)
    bias      = (n_r - 1) * (theta_bar - theta_hat)
    variance  = (n_r - 1) / n_r * np.nansum((theta_j - theta_bar) ** 2, axis=0)
    se        = np.sqrt(np.maximum(variance, 0.0))

    eps      = 1e-12
    rel_bias = np.abs(bias) / np.maximum(np.abs(theta_hat), eps)
    rel_se   = se / np.maximum(np.abs(theta_hat), eps)
    ok_se    = rel_se <= 0.05

    n_pass = int(np.sum(ok_se[1:]))
    print(f"\nPositions with rel_SE < 5%: {n_pass}/{N_VEC - 1}")

    # ── Jackknife plots ──────────────────────────────────────────────────────
    valid = ~np.isnan(theta_hat[1:])
    bidx  = np.where(valid)[0] + 1

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].errorbar(bidx, theta_hat[1:][valid], yerr=se[1:][valid],
                     fmt="o-", lw=1, ms=4, color="steelblue", ecolor="lightsteelblue")
    axes[0].set_title("Volume ratios with jackknife SE")
    axes[0].set_xlabel("Bin boundary i")
    axes[0].set_ylabel("V[i−1]/V[i]")
    axes[0].grid(alpha=0.3)

    colors = np.where(ok_se[1:][valid], "steelblue", "tomato")
    for xi, yi, c in zip(bidx, rel_se[1:][valid], colors):
        axes[1].bar(xi, yi, color=c, alpha=0.8, width=0.7)
    axes[1].axhline(0.05, color="black", ls="--", lw=1, label="5% threshold")
    axes[1].set_title("Relative SE per bin (blue=pass, red=fail)")
    axes[1].set_xlabel("Bin boundary i")
    axes[1].set_ylabel("Rel SE")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / "03_jackknife.png", dpi=150)
    plt.show()

    # ── Save jackknife result ────────────────────────────────────────────────
    stamp_jk = _time.strftime("%Y%m%d_%H%M%S")
    jk_npz   = RESULT_DIR / f"tyson_jackknife_{stamp_jk}.npz"
    np.savez_compressed(
        jk_npz,
        all_volume_ratios = mat,
        theta_hat         = theta_hat,
        se                = se,
        bias              = bias,
        rel_bias          = rel_bias,
        rel_se            = rel_se,
        ok_se             = ok_se,
        n_runs            = np.array(N_RUNS),
        seeds             = np.arange(SEED0, SEED0 + N_RUNS),
    )
    print(f"Saved jackknife result: {jk_npz}")